# ISLES26 — Pipeline Smoke Tests
Runs the full test suite inline on Kaggle without requiring pytest CLI.
Each section maps to one test class. All critical data transformation
stages are exercised before any training is launched.

**Expected runtime:** ~3–5 minutes (CPU). Track C LLM tests are skipped
unless `RUN_LLM_TESTS=True` (requires GPU + transformers installed).

**Mount:** Attach your pipeline scripts as a Kaggle Dataset or copy them
into `/kaggle/working/pipeline/` before running.

In [ ]:
import subprocess, sys

# Install test dependencies if not already present
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pytest', 'pytest-tb', '-q'],
               capture_output=True)

# Flag: set True only on GPU instances with transformers + Qwen2.5-1.5B available
RUN_LLM_TESTS = False
print('Setup complete.')

In [ ]:
import sys
from pathlib import Path

# ── Adjust this path to wherever the pipeline scripts are mounted ─────────────
PIPELINE_DIR = Path('/kaggle/working/pipeline')
assert PIPELINE_DIR.exists(), (
    f'Pipeline dir not found: {PIPELINE_DIR}\n'
    'Copy or mount the pipeline/ folder before running.'
)
sys.path.insert(0, str(PIPELINE_DIR))
TEST_FILE = PIPELINE_DIR / 'tests' / 'test_pipeline.py'
assert TEST_FILE.exists(), f'Test file not found: {TEST_FILE}'
print(f'Pipeline dir: {PIPELINE_DIR}')
print(f'Test file   : {TEST_FILE}')

## Run all tests

In [ ]:
import pytest

# Deselect LLM forward tests unless explicitly enabled
deselect_args = (
    [] if RUN_LLM_TESTS
    else ['--deselect', str(TEST_FILE) + '::TestConditioning::test_track_c_returns_llm_conditioner',
          '--deselect', str(TEST_FILE) + '::TestIntegration::test_track_switch_same_input']
)

exit_code = pytest.main([
    str(TEST_FILE),
    '-v',
    '--tb=short',          # compact tracebacks
    '--no-header',
    '-p', 'no:warnings',
] + deselect_args)

print(f'\nTest suite exit code: {exit_code}')
if exit_code == 0:
    print('✓ All tests passed.')
else:
    print('✗ Some tests failed — review output above before proceeding to training.')

## Run a specific test group
Useful for debugging a specific stage without running the full suite.

In [ ]:
# Change the -k value to target a specific class or test:
#   'preprocessing'  → TestPreprocessing
#   'metadata'       → TestMetadataEncoding
#   'augmentation'   → TestAugmentation
#   'dataset'        → TestDataset
#   'splits'         → TestSplits
#   'conditioning'   → TestConditioning
#   'model'          → TestModel
#   'loss'           → TestLoss
#   'evaluate'       → TestEvaluate
#   'integration'    → TestIntegration

TARGET = 'model'   # ← edit this

pytest.main([
    str(TEST_FILE),
    '-v',
    '--tb=long',
    '-k', TARGET,
    '-p', 'no:warnings',
])